# Mapping Excel to SPOD
- Prerequisites: 
  - Anaconda packages: `pandas, openpyxl`

This script imports the Mapping Excel sheet and processes it into 
SPOD JSON format, which can be imported back.

## Excel Structure Requirements

Sheet with name 'Mapping' containing the [Table](https://support.microsoft.com/en-us/office/create-and-format-tables-e81aa349-b006-4f8a-9806-5af9df0ac664) 'mapping'.

The mapping table must containt the columns:

FQD -> IM:Entity + ':' IM:Attribute


## Result

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet

## Configuration
The following parameters has to be definded when running as regular python script

In [ ]:
#MODEL = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.import.regen.json'
MODEL = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
#MODEL = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.regen.json'

#MAPPING = 'testdata/Sika_Mapping_DE-2022-05-09.xlsx'
#MAPPING = 'testdata/2022-06-02 Mapping France.xlsx'
MAPPING = 'testdata/Sika_Mapping_DE-2022-05-09TE.xlsx'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import shutil
import copy
import time
from pathlib import Path

In [ ]:
# openpyxl
import openpyxl
from openpyxl.worksheet.table import Table
from openpyxl.utils import cell
from openpyxl.styles import PatternFill

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
import jsonpath_ng as jsonpath

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/excel-mapping-{run_stamp}.log'

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")

file_handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger('ExcelMappingLogger')
logger.setLevel(logging.DEBUG)

root = logging.getLogger()
root.addHandler(file_handler)
root.addHandler(console_log_handler)

business_logger = logging.getLogger('business')
business_logger.setLevel(logging.DEBUG)
business_logger.info('Starting busines logging')

In [ ]:
spod_file = Path(MODEL)
assert spod_file.is_file(), f"Cannot find SPOD file '{spod_file.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
logger.info(f"Loaded SPOD {spod['model']['name']} from '{spod_file.resolve()}' revision {spod['_imprint_'].get('git-revision','???')}")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

## Use the fyayc SPOD library

### Tools path

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

In [ ]:
from SSOT_infra import logmessages

tool_logger = logging.getLogger('tool')
tool_logger.setLevel(logging.DEBUG)
tool_logger.info('Starting tool logging')

def tool_logging(message: str):
    tool_logger.debug(message)

logmessages.logtrap = tool_logging

In [ ]:
from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

# Changes in mapping.xlsx versus previous version
Compare new mapping.xlsx with previous version and update the SPOD accordingly.
Create a log of changes in JSON format, close to the SPOD structure.

In [ ]:
source_excel_file = Path(MAPPING)
logger.debug(f"Loading workbook {source_excel_file.resolve()}")
wb = openpyxl.load_workbook(source_excel_file)

In [ ]:
sheetname = 'Mapping'
assert wb[sheetname] is not None, f"No sheet named '{sheetname}' in workbook '{source_excel_file}'"
logger.info(f"Loaded workbook {MAPPING} containing sheet '{sheetname}'")

## Create a duplicat to annotate import progress

In [ ]:
import_result_excel = source_excel_file.with_stem(source_excel_file.stem + '_imported')
shutil.copy(source_excel_file, import_result_excel)
wb_report = openpyxl.load_workbook(import_result_excel)
logger.info(f"Writing import report to '{import_result_excel}'")

# Compare mapping sheet against SPOD 

In [ ]:
mappings = wb['Mapping']
logger.info(f"Sheet {mappings} contains tables: {list(mappings.tables.keys())}")
mapping_table = mappings.tables['mapping']

table_range_tuple = cell.range_boundaries(mapping_table.ref)
logger.info(f"Processing table '{mapping_table.displayName}' spanning {mapping_table.ref} with headers {mapping_table.headerRowCount} {table_range_tuple}")

first_data_row = table_range_tuple[1] + mapping_table.headerRowCount
headers = mappings[table_range_tuple[1]]

In [ ]:
mapping_report_sheet = wb_report['Mapping']

In [ ]:
column_mapping = dict()

index = 0
for column in headers:
    name = column.value
    hits = list(filter(lambda t: t[1]['name'] == name, spod['systems'].items()))
    if len(hits) == 1:
        skey, system = hits[0]
        logger.debug(f"Header '{name}' matches system {skey} {system['name']}")
    else:
        if name.strip() not in ['FQN', 'EID', 'AID', '#', 'Examples', 'Description'] \
                and not name.startswith('Entity (') and not name.startswith('Attribute (') \
                and not name.startswith('INTF'):
            values = mappings[index + 1]
            logger.warning(f"Column '{name}' missing system! {len(values)}")
    column_mapping[name] = index
    index += 1

list(column_mapping.items())[:18], len(column_mapping)

In [ ]:
column_mapping.get('Jedele')

In [ ]:
present = '\x1b[32mpresent\x1b[39m'
absent = '\x1b[31mabsent\x1b[39m'
for skey, system in spod['systems'].items():
    print(f"System {system['name']} is " \
          f"{present if column_mapping.get(system['name']) is not None else absent}")

## jspath templates to access attributes

In [ ]:
columns = { 'Examples': '$.attributes["{attr}"].examples["en"]' }

for skey, system in spod['systems'].items():
    columns[system['name']] = '$.columns["{col}"]["name"]'

In [ ]:
columns

In [ ]:
def build_json_path(path: str, **kwargs) -> jsonpath.Child:
    expanded = path.format(**kwargs)
    path = jsonpath.parse(expanded)
    return path

In [ ]:
path = build_json_path(columns['Examples'], attr='ATTR249')
path

In [ ]:
r = path.find(spod)
if len(r) > 0:
    r[0].value

In [ ]:
def get_aid_from_fqdn(value: str) -> str:
    if value is not None:
        if ':' in value:
            index = value.rfind(':')
            return value[index+1:]
        return value
    return None

In [ ]:
def process_row(headers: tuple, row: tuple):
    attrid = get_aid_from_fqdn(row[column_mapping['AID']])
    for header in headers:
        index = header.col_idx - 1
        new_value = row[index]
        
        colmapping = columns.get(header.value)
        if colmapping is not None:
            jpath = build_json_path(colmapping, attr=attrid)
            hits = jpath.find(spod)
            
            # accept empty
            if len(hits) > 0 and new_value is not None:
                assert len(hits) == 1, f"Expecting only one precise hit, got: {hits}"
                current_value = hits[0].value
                if current_value != new_value:
                    logger.info(f"Updating current value '{current_value}' to '{new_value}'")
                    jpath.update(spod, new_value)
            else:
                logger.debug(f"Cell and new value are undefined")
        else:
            logger.debug(f"Column {header.value} = {new_value}")
        


In [ ]:
sample_row = next(mappings.iter_rows(min_row=first_data_row, max_row=first_data_row + 1, values_only=True))
#process_row(headers, sample_row)
sample_row

In [ ]:
mp = Path(MAPPING)

create_statement = f"Import of {mp.name}"
user = 'bue'
from datetime import datetime
import pytz

timestamp_now = datetime.now()
timestamp_now_utc = pytz.utc.localize(timestamp_now)

stamp = '2022-04-28 15:38:01 UTC'
change_stamp = '2022-04-28 15:38:01.1'
stamp_format = '%Y-%m-%d %H:%M:%S.%f %Z'

stamp = timestamp_now_utc.strftime('%Y-%m-%d %H:%M:%S %Z')
change_stamp = timestamp_now_utc.strftime(stamp_format)

In [ ]:
logger.info(f"Using timestamp {change_stamp} for changes")

In [ ]:
SOURCE_KEY = 'Excel-Mapping-Import-Notebook'

In [ ]:
def neq(lhs, rhs) -> bool:
    if lhs is None and rhs is None:
        return False
    if lhs == rhs:
        return False
    if lhs is None and (isinstance(rhs, str) and len(rhs) < 1):
        return False
    if rhs is None and (isinstance(lhs, str) and len(lhs) < 1):
        return False
    return True

def update_column(column, name: str, tech: str, row: tuple, attributes: set, mandatory) -> []:
    result = None
    messages = []
    updated = copy.deepcopy(column)

    current_attributes = set(column['attributesmapped'])
    diff = current_attributes.symmetric_difference(attributes)
    if len(diff) > 0:
        updated['attributesmapped'] = list(attributes)
        result = updated
        messages.append(f"Altered attribute mapping from {current_attributes} to {attributes}")
        
    if neq(name, column['name']) or neq(tech, column['database_col_id']):        
        updated['name'] = name
        updated['um'] = user
        updated['dm'] = stamp
        result = updated
        messages.append(f"Renamed from {column['name']} to {name}")

    prev = updated.get('mandatory')
    if mandatory is not None and mandatory != prev:
        updated['mandatory'] = mandatory
        messages.append(f"Changed 'mandatory' from {prev} to {mandatory}")
        result = updated
        
    assert updated['userdefprops'] == column['userdefprops']
    
    return result, messages

In [ ]:
def add_defaults(element: dict):
    values = { 'um': None, 'dm': None, 'minzoomlevel': None, 'maxzoomlevel': None, 'publstatus': None,
             'referencedby': []}
    element.update(values)
    return element

In [ ]:
from SSOT_db.IM_JSON.jsattribute import attr2js
from SSOT_db.IM_JSON.jscolumn import columns2js
from SSOT_db.IM_JSON.jstable import tables2js

In [ ]:
d3cell = mappings['H12']
print(d3cell)
color_in_hex = d3cell.fill.start_color.index
print(f"{d3cell.value} {color_in_hex} {d3cell.style}")

In [ ]:
def negative_number_generator() -> int:
    counter = 0
    while True:
        counter -= 1
        yield counter
    
new_element_id_generator = negative_number_generator()

default_domain = next(filter(lambda d: next(iter(d[1]['name'].values())) == 'Unknown', spod['domains'].items()))[0]

assert default_domain is not None
print(f"Default domain is {default_domain}")

ual = list(filter(lambda t: t[1]['shortname'] == 'unassigned', spod['entities'].items()))
assert len(ual) == 1, "Expecting exactly one entity named 'unassigned'"
default_entity = ual[0][0]

In [ ]:
new_table_cache = dict()
new_column_cache = dict()
new_attribute_cache = dict()

def process_system_row(spod: dict, system_key: str, key, name, tech, row, mandatory: bool = None):
    column_key = None
    column = None
    
    result = []
    
    # Mapped columns on this line
    aonline = list(filter(lambda cell: cell.value is not None, row[5:]))
    nonempty_cells_values = list(map(lambda c: c.value, aonline))
 
    if key is not None:
        index = key.rfind(':')
        colkey = key[index+1:]
        column = spod['columns'].get(colkey)
    
    column_by_name = None
    if column is None and name is not None and len(name) > 0:
        hits = list(filter(lambda c: c[1]['database-id+'] == system_key and c[1]['name'] == name, 
                           spod['columns'].items()))
        if len(hits) == 1:
            column = hits[0][1]
            colkey = hits[0][0]
        else:
            assert len(hits) == 0, f"Found several matching columns for name {name} in database {key}! {hits}"
    
    if column is not None and column_by_name is not None and column_by_name != column:
        logger.error(f"Column reference and name error! {column} vs {column_by_name}")
    
    attributes = set()
    if column is not None and len(column['attributesmapped']) > 0:
        attributes = attributes | set(column['attributesmapped'])
    
    attr_ref = row[0].value
    if attr_ref is not None and ':' in attr_ref:
        index = attr_ref.rfind(':')
        attr_key = attr_ref[index+1:]
        if attr_key not in attributes:
            attr = spod['attributes'].get(attr_key)
            if attr is not None:
                attributes.add(attr_key)
                logger.debug(f"Using attribute {attr_key} '{attr['techname']}' '{attr['name'].get('en')}' for row {row[0].row}. Line: {nonempty_cells_values}")    
    
    if len(attributes) < 1:
        ename = row[1].value
        aname = row[2].value
        if ename is not None and aname is not None and len(ename) > 0 and len(aname) > 0:
            logger.warning(f"No attribute found. Using name lookup: {ename}.{aname}")
            candidate_attributes = filter(lambda t: t[1]['name']['de'] == aname, spod['attributes'].items())
            for akey, attr in candidate_attributes:
                ekey = attr['entity']
                entity = spod['entities'][ekey]
                if entity['name']['de'] == ename:
                    attributes.add(akey)
                    logger.debug(f"Sucessfully resolved attribute {ename}.{aname} to {akey}")
    
    if len(attributes) < 1 and len(aonline) > 1:
        """Create a new attribute for the line"""
        aname =  f"R{row[0].row}"
        existing_row_attribute = list(filter(lambda t: t[1]['techname'] == aname or t[1]['name']['de'] == aname, 
                                             spod['attributes'].items()))
        if len(existing_row_attribute) > 0:
            attr_id, _ = existing_row_attribute[0]
            logging.debug(f"Using helper attribute {attr_id}")
        else:
            attr_id = new_attribute_cache.get(aname)
            if attr_id is None:
                description = f"Placeholder attribute for row {row[0].row} by system {spod['systems'][system_key]['name']}: {nonempty_cells_values}"
                logging.warning(f"No attribute mapping found for row {row[0].row} -> create unassigned attribute '{aname}'. Line: {nonempty_cells_values}")
                attr_id = 'ATTR' + str(next(new_element_id_generator))
                line_attribute = attr2js(None)
                line_attribute['techname'] = aname
                line_attribute['name'] = { k: aname for k in spod['languages'].keys() }
                line_attribute['tooltip'] = { k: '' for k in spod['languages'].keys() }
                line_attribute['descr'] = { k: description for k in spod['languages'].keys() }
                line_attribute['entity'] = default_entity
                line_attribute['domain'] = default_domain
                line_attribute['columnsmapped+'] = {} # clear default
                line_attribute['sourceref'] = {} # clear default
                line_attribute['examples'] = []
                line_attribute['userdefprops'].clear()
                new_attribute_cache[aname] = attr_id
                result.append( ('c', attr_id, line_attribute, [description], None) )
            attributes.add(attr_id)
           
    if column is not None:
        updated, messages = update_column(column, name, tech, row, attributes, mandatory)
            
        if updated is not None:
            assert len(messages) > 0, f"No change message for update on {column} from row {row[0].row}"
            logger.debug(f"Found column reference for update: {colkey}")
            logger.debug('\n'.join(messages))
            result.append( ('u', colkey, updated, messages, column) )
    else:
        if name is not None or tech is not None:
            logging.debug(f"Creating new column '{name}' for system '{system_key}")
            if name is None:
                name = tech
            tables = spod['systems'][system_key]['tables+']
            if len(tables) > 0:
                logging.debug("Using first table to add new columns")
                table_id = tables[0]
            else:
                table_id = new_table_cache.get(system_key + '-new')
                if table_id is None:
                    table_id = 'TABL' + str(next(new_element_id_generator))
                    new_table = tables2js(pemptymodel=True)
                    new_table['name'] = 'main'
                    new_table['descr'] = create_statement
                    new_table['uc'] = user
                    new_table['dc'] = stamp
                    new_table['database-id'] = system_key
                    new_table['database-id+'] = system_key  # debugging
                    
                    new_table['prefix'] = None
                    new_table['entitiesmapped'] = []
                    new_table['relationsmapped'] = []
                    new_table['userdefprops'] = {}
                    
                    x = { 'name': 'main', 'database-id': system_key,
                                 'entitiesmapped': [],
                                 'relationsmapped': [],
                                 'columnsmapped': [],
                                 'referencedby': [],
                                 'prefix': None, 'descr': create_statement,
                                 'userdefprops': {},
                                 'uc': user, 'dc': stamp,
                                 # hack
                                 'database-id+': system_key,
                                }
                    add_defaults(new_table)
                    new_table_cache[system_key + '-new'] = table_id
                    result.append( ('c', table_id, new_table, [f"New table {table_id}"], None) )
        
        
            tkey = f'{table_id}:{name}'
            column, prow = new_column_cache.get(tkey, (None, None))
            if column is None:
                column = columns2js(pemptymodel=True)
                column['name'] = name
                column['table-id'] = table_id
                column['database_col_id'] = tech
                column['descr'] = create_statement + f" row:{row[0].row}"
                column['uc'] = user
                column['dc'] = stamp

                column['attributesmapped'] = list(attributes)
                column['domain'] = default_domain
                
                column['R/W'] = 'R'
                column['datatype'] = 'unknown'
                column['format'] = None
                column['userdefprops'] = {}
                
                x = { 'name': name, 'table-id': table_id, 'database_col_id': tech,
                        'uc': user, 'dc': stamp,
                        'attributesmapped': list(attributes),
                        'R/W': 'R',
                        'mandatory': False, 'datatype': 'unknown', 'format': None, 'domain': default_domain, 
                        'descr': create_statement + f" row:{row[0].row}",
                    }
                add_defaults(column)
                
                if mandatory is not None:
                    column['mandatory'] = mandatory
                
                new_column_cache[tkey] = (column, row[0].row)
                msg = f"New column '{name}' mapped to attributes {attributes} from row {row[0].row}"
                result.append( ('c', 'COLU' + str(next(new_element_id_generator)), column, [msg], None) )
                logging.info(msg)
            else:
                msg = f"Cell '{tkey}' is a second mapping of {prow} on row {row[0].row}. Adding attributes {attributes} to column: {column}"
                logging.warning(msg)
                column['attributesmapped'].extend(attributes)
                    
    for t in result:
        type_id = t[1][:4]
        # Crucial part! Generate a unique id for each and every element
        srcname = f"EMI:{source_excel_file.stem.replace(' ','_')}:{row[0].row} {t[1]}"
        row_stamp = datetime.now()
        row_stamp_utc = pytz.utc.localize(row_stamp)
        unique = row_stamp_utc.strftime(stamp_format)
        element = t[2]
        ref = element.get('sourceref')
        if ref is None:
            element['sourceref'] = { SOURCE_KEY: [ srcname, unique ] }
        else:
            ref[SOURCE_KEY] = [ srcname, unique ]
    
    return result

In [ ]:
processed = []
all_changes = []

# TODO switch from system scan to column scan

for skey, system in spod['systems'].items():
    name_col_index = column_mapping.get(system['name'])
    if name_col_index is not None:
        key_col_index = column_mapping.get(skey + ':FQN')
        tech_col_index = column_mapping.get(skey + ':REF')

        data_iterator = mappings.iter_rows(min_row=first_data_row, values_only=False)
        changes = []
        for row in tqdm(data_iterator, desc=f"Processing rows of system {system['name']}", unit=" Row", dynamic_ncols=True):
            value = row[name_col_index].value
            key = None if key_col_index is None else row[key_col_index].value
            tech = None if tech_col_index is None else row[tech_col_index].value
            
            # sika: optional columns are marked grey (0)
            mandatory = None
            color = row[name_col_index].fill.start_color.index
            if color == 0:
                logging.debug(f"Column {value} on system {system['name']} is optional")
                mandatory = False
            else:
                mandatory = True
            
            elements = process_system_row(spod, skey, key=key, name=row[name_col_index].value, tech=tech, row=row, mandatory=mandatory)
            changes += elements
        if len(changes) > 0:
            logging.info(f"Applying {len(changes)} element changes for system {system['name']}")
            all_changes += changes
        processed.append(f"{system['name']} [{skey}]")
    else:
        logging.warning(f"System '{system['name']}' not present")

In [ ]:
len(all_changes), len(processed), processed

## Add new systems

### Fix uniqueness of sourceref entries

# Apply changes to spod

In [ ]:
changes = []
creations = 0
updates = 0

spod_updated = copy.deepcopy(spod)
with tqdm(total=len(all_changes), unit=' Operation') as progress:
    for action, key, element, messages, previous in all_changes:
        progress.set_description(f"Processing {action} on {key}")
        change = { 
            'action': action, 
            'key': key, 
        }
        message = f"({action} on {key}"
        if action == 'c':
            change['new'] = element
            if key.startswith('COLU'):
                message = f"Creating column {key}"
                spod_updated['columns'][key] = element
                creations += 1
            if key.startswith('TABL'):
                message = f"Creating table {key} for system '{element['name']}' {element['database-id']}"
                spod_updated['tables'][key] = element
                creations += 1
            if key.startswith('ATTR'):
                message = f"Creating attribute {key} for entity '{element['entity']}'"
                spod_updated['attributes'][key] = element
                creations += 1
                
        if action == 'u':
            change['new'] = element
            change['current'] = previous
            if key.startswith('COLU'):
                message = f"Updating column {key}"
                spod_updated['columns'][key] = element
                updates += 1
                
        logging.debug(message)
        change['message'] = message
        change['changes'] = messages
        changes.append(change)
        progress.update(1)
        time.sleep(0.003)

In [ ]:
creations, updates

In [ ]:
changelog_file = Path('changelog.json') 
with open(changelog_file, 'w') as out:
    json.dump(changes, out)
logger.info(f"Wrote {len(changes)} changes to {changelog_file}")

### Upgrade spod structure

In [ ]:
# Patch JSON
#spod_updated['actorroles'] = {}

In [ ]:
# fix SPOD
#spod_updated['entities'].pop('ENTI116', None)
#spod_updated['entities'].pop('ENTI364', None)
#ATTR397
#ATTR4700
#ATTR1309

## Store changed SPOD 

In [ ]:
updated = Path(MODEL).with_suffix('.import.json')
with open(updated, 'w') as out:
    json.dump(spod_updated, out, indent=4)
print(f"Wrote {updated}")

# Try to merge

In [ ]:
from LOAD_MODELS.LOAD_INFRA import mergedbs
from SSOT_db.SQL_INFRA import dbConnect
from SSOT_db.IM_JSON.jsbase import JSModel

In [ ]:
sourcedb = Path(MODEL).with_suffix('.db')
assert sourcedb.is_file(), f"Source database '{sourcedb} is missing'"

tempdb = sourcedb.with_suffix('.import.db')
# clean slate
tempdb.unlink(missing_ok=True)

shutil.copy(sourcedb, tempdb)
assert tempdb.is_file()

In [ ]:
mappingfile = Path(MAPPING)

In [ ]:
from SSOT_infra import parameters
spod_updated['_imprint_']['git-revision'] = parameters.read_git_description(sourcedb.parent)
new_model = JSModel(spod_updated)

In [ ]:
print(f"Merging updated json into existing DB")

In [ ]:
from SSOT_infra import parameters
spod_updated['_imprint_']['git-revision'] = parameters.read_git_description(sourcedb.parent)
new_model = JSModel(spod_updated)
#mergedbs.checkjsonmodel(pmodel=new_model, pverbose=True)

In [ ]:
print(f"Merging updated json {updated} into existing DB {sourcedb} via updating {tempdb}")

In [ ]:
from contextlib import closing

with closing(dbConnect.openDB(tempdb)):
    logger.info(f"Merging JSON into database {tempdb} for source {SOURCE_KEY}")
    mergeresult = mergedbs.mergejson2sql(new_model, psrcname=SOURCE_KEY, pverbose=True, pcheckonly=False)

mergeresult

In [ ]:
mergeresult.write_json('mergeresult.json')

## Load restore json from updated DB

In [ ]:
from SSOT_db.IM_JSON.jsmodel import sql2json
from SSOT_infra.parameters import read_git_description

with closing(dbConnect.openDB(tempdb)):
    loadedjson = JSModel(pmodel=sql2json(pdbname=str(tempdb)))
loadedjson.jsmodel['_imprint_']['git-revision'] = read_git_description(sourcedb.parent)

In [ ]:
regen_file = tempdb.with_suffix('.regen.json')
with open(regen_file, 'w') as out:
    json.dump(loadedjson.jsmodel, out, indent=4)
logger.info(f"Wrote regenerated JSON to {regen_file}")

In [ ]:
spod_re = loadedjson.jsmodel
for category in spod_re.keys():
    if category in ['entities', 'attributes', 'systems', 'tables', 'columns']:
        print(f"Count difference for '{category}':  {len(spod[category])} -> {len(spod_re[category])}")
        for key, element in spod_re[category].items():
            pass